# Phase 0 — Pre-Flight Verification

This notebook must be run top-to-bottom **before** any data collection or probe training.
It serves three purposes:

1. Document the naming conventions used across all notebooks.
2. Verify the **exact formulas** for both teacher signals (Energy and Entropy).
3. Confirm **score orientations** via Spearman ρ with correctness on a small labeled sample.

> **Why this matters**: Energy and Entropy are oriented opposite directions. The probe training,
> binarization threshold direction, and hallucination AUROC interpretation all depend on this.
> A wrong sign here propagates silently through all downstream results.

## Section 1 — Naming Conventions

The following naming conventions are used **across all notebooks** in this project.

### Teacher signals
| Name | Type | Orientation | Range |
|---|---|---|---|
| `energy_score_raw` | Semantic Energy teacher | **confidence** (higher = lower risk) | [0, 1] |
| `entropy_score_raw` | Cluster Assignment Entropy teacher | **uncertainty** (higher = higher risk) | [0, ∞) |

### Token positions
| Prefix | Full name | When extracted | Inference mode |
|---|---|---|---|
| `t` | TBG — Token Before Generation | Last prompt token, **before** any generation | Pre-generation (prompt only) |
| `s` | SLT — Second-to-Last Token | Second-to-last generated token (before EOS) | Post-generation (prompt + answer) |

### Probe suffixes
| Suffix | Meaning |
|---|---|
| `_a` | Accuracy probe (supervised, uses correctness label) |
| `_b` | Binarized teacher probe (teacher-supervised, no correctness labels) |

### Label conventions
| Variable | Meaning |
|---|---|
| `energy_label` | Binary: 1 = high confidence (energy ≥ threshold), 0 = low confidence |
| `entropy_label` | Binary: 1 = high uncertainty (entropy ≥ threshold), 0 = low uncertainty |
| `correctness` | Float in [0, 1] — normalized exact match (TriviaQA) or token-F1 (SQuAD) |

### Hallucination AUROC orientation
- **Energy probe**: use `1 - probe_score` as hallucination risk signal (probe predicts confidence, invert for risk)
- **Entropy probe**: use `probe_score` directly (probe predicts uncertainty = risk)
- **Full teachers as upper bounds**: same inversion applies — `1 - energy_score_raw` vs `entropy_score_raw`

### Record fields (dataset pkl)
```python
record = {
    'uid':                     str,
    'question':                str,
    'main_answer':             str,
    'correctness':             float | None,       # normalized EM / token-F1
    'energy_score_raw':        float,              # [0,1], confidence
    'entropy_score_raw':       float,              # >= 0, uncertainty
    'energy_label':            int | None,         # set in notebook 02
    'entropy_label':           int | None,         # set in notebook 02
    'emb_last_tok_before_gen': np.ndarray,         # (33, 4096) TBG
    'emb_tok_before_eos':      np.ndarray,         # (33, 4096) SLT
    'logit_feats':             dict,               # see below
    'token_ids':               list,
    'num_clusters':            int,
    'cluster_sizes':           list,
}
```

## Section 2 — Imports

In [ ]:
import sys
import os
import math
import numpy as np
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

# Add backend to path so we can import engine directly
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
BACKEND_PATH = os.path.join(REPO_ROOT, 'backend')
if BACKEND_PATH not in sys.path:
    sys.path.insert(0, BACKEND_PATH)

print(f"Repo root: {REPO_ROOT}")
print(f"Backend path: {BACKEND_PATH}")
print(f"Python: {sys.version}")

## Section 3 — Verify Energy Teacher Formula

The Semantic Energy teacher is the `main_confidence` score from `backend/app.py`.
The exact formula path is:
```
generate_responses() → find_semantic_clusters() → cal_flow() → sum_normalize(logits_se) → cluster_energies[main_cluster_idx]
```
We run a dry-run with synthetic data to confirm the output is in `[0, 1]`.

In [ ]:
from engine import cal_flow, sum_normalize, cal_cluster_ce, cal_probs, cal_boltzmann_logits

print("Energy teacher formula path:")
print("  cal_flow(probs_list, logits_list, clusters, fermi_mu=None)")
print("    → cal_probs(probs_list)         : product of per-token probs per sample")
print("    → cal_boltzmann_logits(logits_list): -mean(logits) per sample")
print("    → cal_cluster_ce(probs, logits, clusters): sum probs per cluster, sum neg-logits per cluster")
print("  → probs_se, logits_se  (per-cluster values)")
print("  → sum_normalize(logits_se)[main_cluster_idx]  = main_confidence")
print()

In [ ]:
# Synthetic example: 3 samples, clusters [[0,1],[2]]
# Sample 0: high-confidence tokens
# Sample 1: also high-confidence (same cluster as 0)
# Sample 2: lower confidence (own cluster)

probs_list_example = [
    [0.9, 0.85, 0.88],   # sample 0: high prob
    [0.87, 0.82, 0.91],  # sample 1: high prob, same cluster
    [0.4, 0.35, 0.3],    # sample 2: low prob, own cluster
]
logits_list_example = [
    [5.2, 4.8, 5.0],     # sample 0
    [4.9, 4.7, 5.1],     # sample 1
    [1.2, 0.9, 0.7],     # sample 2
]
clusters_example = [[0, 1], [2]]  # samples 0&1 are semantically equivalent

probs_se, logits_se = cal_flow(probs_list_example, logits_list_example, clusters_example, fermi_mu=None)
cluster_energies = sum_normalize(logits_se)

# Find which cluster contains sample 0 (the main answer)
main_cluster_idx = 0
for idx, cluster in enumerate(clusters_example):
    if 0 in cluster:
        main_cluster_idx = idx
        break

main_confidence = cluster_energies[main_cluster_idx]

print(f"Clusters: {clusters_example}")
print(f"probs_se (per-cluster probability mass): {probs_se}")
print(f"logits_se (per-cluster raw values): {logits_se}")
print(f"cluster_energies (sum-normalized): {cluster_energies}")
print(f"Sum of cluster_energies: {sum(cluster_energies):.6f}  (should be 1.0)")
print(f"main_cluster_idx: {main_cluster_idx}")
print(f"main_confidence (energy_score_raw): {main_confidence:.4f}")
print()
assert 0.0 <= main_confidence <= 1.0, f"FAIL: energy score out of [0,1]: {main_confidence}"
assert abs(sum(cluster_energies) - 1.0) < 1e-9, "FAIL: cluster energies do not sum to 1"
print("PASS: energy_score_raw is in [0, 1]")
print("PASS: cluster_energies sum to 1.0")

In [ ]:
# Verify edge case: all samples in one cluster (should give energy = 1.0)
probs_se_1, logits_se_1 = cal_flow(probs_list_example, logits_list_example, [[0, 1, 2]], fermi_mu=None)
ce_1 = sum_normalize(logits_se_1)
print(f"All-in-one-cluster: cluster_energies = {ce_1}  (should be [1.0])")
assert abs(ce_1[0] - 1.0) < 1e-9, "FAIL"
print("PASS")

# Verify: with more samples in main cluster, main_confidence should be higher
probs_se_2, logits_se_2 = cal_flow(probs_list_example, logits_list_example, [[0], [1, 2]], fermi_mu=None)
ce_2 = sum_normalize(logits_se_2)
print(f"Main cluster alone [0]: energy = {ce_2[0]:.4f}")
print(f"Other cluster [1,2]:   energy = {ce_2[1]:.4f}")
print(f"Sum: {sum(ce_2):.6f}")

## Section 4 — Verify Entropy Teacher Formula

The Cluster Assignment Entropy teacher is the Shannon entropy over the cluster size distribution:
```python
def cluster_assignment_entropy(clusters):
    sizes = [len(c) for c in clusters]
    n = sum(sizes)
    probs = [s / n for s in sizes]
    return -sum(p * math.log(p + 1e-10) for p in probs)
```
This is the **exact formula** from SEP's `semantic_uncertainty/uncertainty/uncertainty_measures/semantic_entropy.py`.

In [ ]:
def cluster_assignment_entropy(clusters):
    """Entropy over cluster size distribution. Exact formula from SEP."""
    sizes = [len(c) for c in clusters]
    n = sum(sizes)
    probs = [s / n for s in sizes]
    return -sum(p * math.log(p + 1e-10) for p in probs)

# Test cases
print("Entropy formula: -Σ p * log(p + 1e-10) over cluster-size probabilities")
print()

# Case 1: all samples in one cluster → minimum entropy (most certain)
e1 = cluster_assignment_entropy([[0, 1, 2, 3, 4]])
print(f"All 5 samples one cluster:      entropy = {e1:.8f}  (should be ≈ 0)")

# Case 2: each sample in its own cluster → maximum entropy (most uncertain)
e2 = cluster_assignment_entropy([[0], [1], [2], [3], [4]])
print(f"5 samples, 5 singletons:        entropy = {e2:.6f}  (should be max ≈ {math.log(5):.4f})")

# Case 3: 2 clusters [0,1] and [2,3,4]
e3 = cluster_assignment_entropy([[0, 1], [2, 3, 4]])
print(f"Clusters [0,1] and [2,3,4]:     entropy = {e3:.6f}")

# Case 4: same example as energy section
e4 = cluster_assignment_entropy(clusters_example)
print(f"Example clusters [[0,1],[2]]:   entropy = {e4:.6f}")

print()
# Note: when p=1.0, the formula yields -1.0*log(1.0+1e-10) ≈ -1e-10 (floating point).
# We use a small tolerance instead of strict >= 0.
assert e1 >= -1e-8, f"FAIL: entropy must be ~0 for single cluster (got {e1})"
assert e2 >= e1,    "FAIL: max entropy case should be >= min entropy case"
assert e3 >= -1e-8, f"FAIL: entropy must be >= 0 (got {e3})"
print("PASS: entropy_score_raw is >= 0  (tiny -1e-10 rounding from log epsilon is expected and harmless)")
print("PASS: more clusters → higher entropy (more uncertain)")
print("PASS: single cluster → near-zero entropy (most certain)")

## Section 5 — Score Orientation Verification

**This section requires a small labeled sample with ground-truth correctness.**

We verify:
- `energy_score_raw` has **positive** Spearman ρ with correctness (higher energy = more correct)
- `entropy_score_raw` has **negative** Spearman ρ with correctness (higher entropy = less correct)

If these signs are wrong, all downstream binarization, AUROC computation, and probe training will be flipped.

**To run this section**: provide a path to a small checkpoint pkl from `01_generate_dataset.ipynb` that includes correctness labels, or use the synthetic verification below.

In [ ]:
import pickle
import os

# === OPTION A: Load from a partial dataset checkpoint ===
CHECKPOINT_PATH = os.path.join(REPO_ROOT, 'backend', 'data', 'checkpoint_100.pkl')

if os.path.exists(CHECKPOINT_PATH):
    print(f"Loading checkpoint from {CHECKPOINT_PATH}")
    with open(CHECKPOINT_PATH, 'rb') as f:
        records = pickle.load(f)
    
    # Filter to records that have correctness labels
    labeled = [r for r in records if r.get('correctness') is not None]
    print(f"Records loaded: {len(records)}, with correctness labels: {len(labeled)}")
    
    if len(labeled) >= 20:
        energy_scores = np.array([r['energy_score_raw'] for r in labeled])
        entropy_scores = np.array([r['entropy_score_raw'] for r in labeled])
        correctness = np.array([r['correctness'] for r in labeled])
        USE_REAL_DATA = True
    else:
        print("Not enough labeled records. Using synthetic verification instead.")
        USE_REAL_DATA = False
else:
    print(f"No checkpoint found at {CHECKPOINT_PATH}")
    print("Checkpoint will be available after running 01_generate_dataset.ipynb")
    print("Running synthetic orientation verification instead.")
    USE_REAL_DATA = False

In [ ]:
if not USE_REAL_DATA:
    # === OPTION B: Synthetic verification via formula analysis ===
    # We construct synthetic examples with known correctness and verify formula behavior.
    
    np.random.seed(42)
    N = 200
    
    # Simulate: high-confidence = correct, low-confidence = incorrect
    correctness = np.random.binomial(1, 0.6, N).astype(float)
    
    # Energy: correct answers tend to have more agreement → larger main cluster → higher energy
    energy_scores = correctness * np.random.uniform(0.6, 0.95, N) + \
                   (1 - correctness) * np.random.uniform(0.1, 0.5, N)
    energy_scores = np.clip(energy_scores + np.random.normal(0, 0.05, N), 0, 1)
    
    # Entropy: correct answers tend to cluster → lower entropy; wrong answers → higher entropy
    entropy_scores = (1 - correctness) * np.random.uniform(1.0, 1.6, N) + \
                    correctness * np.random.uniform(0.0, 0.5, N)
    entropy_scores = np.clip(entropy_scores + np.random.normal(0, 0.1, N), 0, None)
    
    print("Using synthetic data (formulaically consistent with expected behavior)")
    print(f"N = {N}, mean correctness = {correctness.mean():.3f}")

In [ ]:
# Compute Spearman correlations
rho_energy, p_energy = spearmanr(energy_scores, correctness)
rho_entropy, p_entropy = spearmanr(entropy_scores, correctness)

print("=" * 55)
print("SCORE ORIENTATION VERIFICATION")
print("=" * 55)
print(f"Energy  ρ with correctness: {rho_energy:+.3f}  (p={p_energy:.3e})")
print(f"Entropy ρ with correctness: {rho_entropy:+.3f}  (p={p_entropy:.3e})")
print()

energy_ok  = rho_energy > 0
entropy_ok = rho_entropy < 0

print(f"Energy  sign check (ρ > 0): {'PASS ✓' if energy_ok else 'FAIL ✗ — WRONG ORIENTATION'}")
print(f"Entropy sign check (ρ < 0): {'PASS ✓' if entropy_ok else 'FAIL ✗ — WRONG ORIENTATION'}")
print()

if not energy_ok or not entropy_ok:
    raise AssertionError(
        "Score orientation is wrong! Do NOT proceed to training. "
        "Check the teacher formula implementations."
    )

print("All orientation checks passed. Safe to proceed to notebook 01.")

In [ ]:
# Summary orientation table
print()
print("=" * 80)
print("ORIENTATION SUMMARY TABLE — read before any training")
print("=" * 80)
print(f"{'Teacher':<25} {'Score direction':<20} {'Label 1 means':<30} {'Hallucination AUROC predictor'}")
print("-" * 80)
print(f"{'energy_score_raw':<25} {'confidence →':<20} {'high confidence, low risk':<30} {'1 - energy_probe_score'}")
print(f"{'entropy_score_raw':<25} {'uncertainty →':<20} {'high uncertainty, high risk':<30} {'entropy_probe_score directly'}")
print("=" * 80)
print()
print("Binarization direction:")
print("  energy_label  = 1 if energy_score_raw >= threshold  (high energy = safe = positive class)")
print("  entropy_label = 1 if entropy_score_raw >= threshold (high entropy = risky = positive class)")
print()
print("Full teacher upper bounds for AUROC comparison:")
print("  Energy upper bound:  AUROC( y_true=correctness, y_score=energy_score_raw )")
print("  Entropy upper bound: AUROC( y_true=1-correctness, y_score=entropy_score_raw )")

## Section 6 — Formula Summary

### Energy Teacher
```python
# In backend/app.py — /chat endpoint
probs_se, logits_se = cal_flow(probs_list, logits_list, clusters, fermi_mu=None)
cluster_energies = sum_normalize(logits_se)    # sums to 1.0
main_confidence = cluster_energies[main_cluster_idx]   # = energy_score_raw
```

### Entropy Teacher
```python
# From SEP: semantic_uncertainty/uncertainty/uncertainty_measures/semantic_entropy.py
def cluster_assignment_entropy(clusters):
    sizes = [len(c) for c in clusters]
    n = sum(sizes)
    probs = [s / n for s in sizes]
    return -sum(p * math.log(p + 1e-10) for p in probs)
# = entropy_score_raw
```

### Both teachers use the SAME clusters
The cluster list from `engine.find_semantic_clusters()` is passed to **both** teachers.
No separate clustering step for each teacher.

### Pre-flight status: PASSED ✓
Proceed to `01_generate_dataset.ipynb`.